# SAR Flood Susceptibility: Classification and Validation

Classifies the Sentinel-1 flood-frequency raster (0-9 detections) into five categories with Jenks natural breaks, aggregates it to hexagons, and validates it against the historical flood reports.

Inputs: the flood-frequency raster exported from Google Earth Engine. Outputs: classified raster and hexagon layers.

In [ ]:
# Standard library: system path handling
import sys
# Timing utilities
import time
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Numerical operations
import numpy as np
# Tabular data handling
import pandas as pd
# Geospatial data handling
import geopandas as gpd
# Raster reading and writing
import rasterio
# Windowed reading to process the raster in strips
from rasterio.windows import Window
# Mask a raster with a polygon
from rasterio.mask import mask as rio_mask
# Natural-breaks (Jenks) classification
import jenkspy
# Plotting library
import matplotlib.pyplot as plt
# Legend patches for the map
from matplotlib.patches import Patch

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)
# Create a logger that writes to the logs folder
logger = get_logger("step11c_sar_classification", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("SAR (local) iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with raw data
raw_dir = resolve_path(config, "raw_dir", base_dir=Path.cwd())
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Folder for map outputs
maps_dir = ensure_dir(PROJECT_ROOT / "outputs" / "maps")

# Path to the flood-frequency raster exported from Google Earth Engine
RUTA_TIF = raw_dir / "sentinel1_flood" / "efci_flood_frequency_2015_2025.tif"
# Check the raster is available locally
assert RUTA_TIF.exists(), f"No se encontró {RUTA_TIF} -- ¿ya se copió el GeoTIFF exportado desde Drive?"
# Print the input path
print("GeoTIFF de entrada:", RUTA_TIF)

## 1. Full histogram of the frequency raster (0-9)

In [ ]:
# Start a timer
t0 = time.time()
# Pixel counts per value (0-255)
counts = np.zeros(256, dtype=np.int64)
# Total number of pixels read
total_pixeles = 0

# Open the raster
with rasterio.open(RUTA_TIF) as src:
    # Keep the raster profile for the output
    perfil_origen = src.profile.copy()
    # Strip height in rows, to limit memory use
    STRIP = 4096
    # Loop over the raster in horizontal strips
    for row_off in range(0, src.height, STRIP):
        # Height of the current strip
        h = min(STRIP, src.height - row_off)
        # Window that covers the strip
        window = Window(0, row_off, src.width, h)
        # Read the strip
        arr = src.read(1, window=window)
        # Count pixels per value
        counts += np.bincount(arr.ravel(), minlength=256)
        # Accumulate the total pixel count
        total_pixeles += arr.size

# Values that actually appear, with their counts
valores_presentes = {v: int(c) for v, c in enumerate(counts) if c > 0}
# Print the total number of pixels and elapsed time
print(f"Total de píxeles: {total_pixeles:,} [{time.time()-t0:.1f}s]")
# Print a header for the value counts
print("Valores presentes (valor: conteo de píxeles):")
# Loop over the values present
for v, c in valores_presentes.items():
    # Print each value, its count and its share
    print(f"  {v}: {c:,}  ({c/total_pixeles*100:.4f}% del AOI)")

# Pixels with at least one flood detection
pixeles_inundados = sum(c for v, c in valores_presentes.items() if v > 0)
# Flooded area in square kilometres (30 m pixels)
area_real_km2 = pixeles_inundados * 30 * 30 / 1e6
# Print the flooded pixels and area
print(f"\nPíxeles con al menos 1 detección de inundación (valor>=1): {pixeles_inundados:,} "
      f"-> área real a 30 m: {area_real_km2:,.1f} km²")

## 2. Jenks breaks (5 categories) on the full histogram

In [ ]:
# Combinatorics utilities
import itertools

# Detection values above zero, sorted
valores = sorted(v for v in valores_presentes if v > 0)
# Pixel count of each value
conteos = [valores_presentes[v] for v in valores]
# Number of distinct values
n = len(valores)

# Within-group cost of a candidate class
def costo_grupo(idx_ini, idx_fin):
    # Values of the class
    vs = np.array(valores[idx_ini:idx_fin+1], dtype=float)
    # Pixel counts of the class, used as weights
    ws = np.array(conteos[idx_ini:idx_fin+1], dtype=float)
    # Weighted mean of the class
    media = np.average(vs, weights=ws)
    # Weighted sum of squared deviations
    return np.sum(ws * (vs - media) ** 2)

# Best cost and best breaks so far
mejor_costo, mejor_cortes = None, None
# Try every way to split the values into five classes
for cortes in itertools.combinations(range(1, n), 4):
    # Class limits for this combination
    limites = [0] + list(cortes) + [n]
    # Total cost of the combination
    costo_total = sum(costo_grupo(limites[i], limites[i+1]-1) for i in range(5))
    # Keep the combination with the lowest cost
    if mejor_costo is None or costo_total < mejor_costo:
        # Remember the best cost and breaks
        mejor_costo, mejor_cortes = costo_total, cortes

# Final class limits
limites = [0] + list(mejor_cortes) + [n]
# Class labels
ETIQUETAS = {1: "Muy Baja", 2: "Baja", 3: "Moderada", 4: "Alta", 5: "Muy Alta"}
# Lookup table from raster value to class
LOOKUP = np.zeros(256, dtype=np.uint8)
# List that collects the class summary
resumen_clases = []
# Loop over the five classes
for i in range(5):
    # First and last value of the class
    ini, fin = limites[i], limites[i+1]-1
    # Values of the class
    vs_grupo = valores[ini:fin+1]
    # Loop over the class values
    for v in vs_grupo:
        # Assign the class to each value in the lookup table
        LOOKUP[v] = i + 1
    # Pixels in the class
    px = sum(valores_presentes[v] for v in vs_grupo)
    # Store the class summary
    resumen_clases.append({
        "categoria": i+1, "etiqueta": ETIQUETAS[i+1], "valores_originales": vs_grupo,
        "n_pixeles": px, "area_km2_nativa": round(px * 30 * 30 / 1e6, 2),
    })

# Class summary table
tabla_clases = pd.DataFrame(resumen_clases)
# Print the weighted cost of the chosen breaks
print(f"Costo (SSD ponderada, menor=mejor): {mejor_costo:.1f}")
# Show the class table
tabla_clases

## 3. Classified raster at native 30 m resolution

In [ ]:
# Path of the classified raster
RUTA_CLASIFICADO = processed_dir / "sar_flood_susceptibility_clasificado.tif"
# Temporary path, to avoid writing partial files to the final folder
RUTA_CLASIFICADO_TMP = Path("/tmp/sar_flood_susceptibility_clasificado.tif")

# Output profile based on the source profile
perfil_out = perfil_origen.copy()
# Use 8-bit integers with compression
perfil_out.update(dtype="uint8", nodata=0, compress="deflate", predictor=2,
                   tiled=True, blockxsize=256, blockysize=256)

# Start a timer
t0 = time.time()
# Open the source raster for reading and the output for writing
with rasterio.open(RUTA_TIF) as src, rasterio.open(RUTA_CLASIFICADO_TMP, "w", **perfil_out) as dst:
    # Strip height in rows
    STRIP = 4096
    # Loop over the raster in strips
    for row_off in range(0, src.height, STRIP):
        # Height of the current strip
        h = min(STRIP, src.height - row_off)
        # Window that covers the strip
        window = Window(0, row_off, src.width, h)
        # Read the strip
        arr = src.read(1, window=window)
        # Write the classified strip using the lookup table
        dst.write(LOOKUP[arr], 1, window=window)
# Print how long the classification took
print(f"Raster clasificado escrito en {time.time()-t0:.1f}s")

# File copy utility
import shutil
# Copy the classified raster to its final location
shutil.copy(RUTA_CLASIFICADO_TMP, RUTA_CLASIFICADO)
# Print the final path
print("Copiado a:", RUTA_CLASIFICADO)

## 4. From raster to polygons: why hexagon aggregation is used

In [ ]:
# Path to the hexagon layer from the spatial analysis
RUTA_HEX_SPATIAL = processed_dir / "hexagonos_h3_res5.geojson"
# Read the hexagons
hex_gdf = gpd.read_file(RUTA_HEX_SPATIAL)
# Print the number of hexagons
print(f"Hexágonos análisis espacial reutilizados: {len(hex_gdf)}")

# Start a timer
t0 = time.time()
# Lists for the zonal statistics of each hexagon
medias, maximos, frac_no0 = [], [], []
# Open the frequency raster
with rasterio.open(RUTA_TIF) as src:
    # Loop over the hexagons
    for idx, row in hex_gdf.iterrows():
        # Hexagon geometry as GeoJSON
        geom = [row.geometry.__geo_interface__]
        # Try the standard quantile cut
        try:
            # Clip the raster to the hexagon
            out_img, _ = rio_mask(src, geom, crop=True, filled=True, nodata=255)
        except ValueError:
            # Hexagon outside the raster: record zeros
            medias.append(0.0); maximos.append(0); frac_no0.append(0.0)
            # Move on to the next unit
            continue
        # Pixel values of the clipped area
        arr = out_img[0]
        # Keep valid pixels only
        valid = arr[arr != 255]
        # Check the hexagon contains valid pixels
        if valid.size == 0:
            # No valid pixels: record zeros
            medias.append(0.0); maximos.append(0); frac_no0.append(0.0)
            # Move on to the next unit
            continue
        # Mean detection frequency
        medias.append(float(valid.mean()))
        # Maximum detection frequency
        maximos.append(int(valid.max()))
        # Share of pixels with at least one detection
        frac_no0.append(float((valid > 0).mean()))

# Store the mean frequency in the hexagon table
hex_gdf["sar_freq_media"] = medias
# Store the maximum frequency
hex_gdf["sar_freq_max"] = maximos
# Store the flooded share
hex_gdf["sar_frac_inundada"] = frac_no0
# Print how long the zonal statistics took
print(f"Estadística zonal por hexágono calculada en {time.time()-t0:.1f}s")
# Print hexagons with a SAR signal
print(f"Hexágonos con señal SAR > 0: {(hex_gdf['sar_freq_media']>0).sum()} / {len(hex_gdf)}")
# Summary statistics of the zonal values
hex_gdf[["h3_cell","sar_freq_media","sar_freq_max","sar_frac_inundada"]].describe()

In [ ]:
# Hexagons with a SAR signal
con_senal = hex_gdf[hex_gdf["sar_freq_media"] > 0]
# Jenks breaks on the hexagon-level mean frequency
breaks_hex = jenkspy.jenks_breaks(con_senal["sar_freq_media"].values, n_classes=5)
# Print the breaks
print("Quiebres Jenks (nivel hexágono, sar_freq_media):", [round(b, 4) for b in breaks_hex])

# Assign a hexagon to a category
def clasificar_hex(v):
    # No signal means category 0
    if v <= 0:
        # Return category 0
        return 0
    # Loop over the five classes
    for i in range(5):
        # Hexagon belongs to the first class whose upper limit it does not exceed
        if v <= breaks_hex[i+1] or i == 4:
            # Return the class
            return i + 1
    # Fallback to the highest class
    return 5

# Apply the category to every hexagon
hex_gdf["categoria"] = hex_gdf["sar_freq_media"].apply(clasificar_hex)
# Labels of the hexagon categories
ETIQUETAS_HEX = {0: "Sin señal SAR", 1: "Muy Baja", 2: "Baja", 3: "Moderada", 4: "Alta", 5: "Muy Alta"}
# Add the label to each hexagon
hex_gdf["etiqueta"] = hex_gdf["categoria"].map(ETIQUETAS_HEX)

# Reproject to a metric CRS to measure area
hex_utm = hex_gdf.to_crs(config["project"]["crs"] if False else "EPSG:32717")
# Hexagon area in square kilometres
hex_gdf["area_km2"] = hex_utm.geometry.area / 1e6

# Summary by category (hexagon count and area)
resumen_hex = hex_gdf.groupby(["categoria","etiqueta"]).agg(
    n_hexagonos=("h3_cell","count"), area_km2=("area_km2","sum")
).reset_index()
# Print an empty line
print()
# Print the summary
print(resumen_hex)

# Path of the hexagon output
RUTA_HEX_OUT = processed_dir / "sar_flood_susceptibility_hexagonos.gpkg"
# Temporary path
RUTA_HEX_TMP = Path("/tmp/sar_flood_susceptibility_hexagonos.gpkg")
# Write the hexagons to a temporary GeoPackage
hex_gdf.to_file(RUTA_HEX_TMP, driver="GPKG")
# File copy utility
import shutil
# Copy the file to its final location
shutil.copy(RUTA_HEX_TMP, RUTA_HEX_OUT)
# Print the saved path
print("\nGuardado:", RUTA_HEX_OUT)

## 5. Validation against historical flood events

In [ ]:
# Path of the event layer
RUTA_EVENTOS = processed_dir / "hazard_event_geo.gpkg"
# Read all events
eventos = gpd.read_file(RUTA_EVENTOS)
# Keep flood events only
eventos_flood = eventos[eventos["hazard_type"] == "flood"].copy()
# Print the number of flood events
print(f"Eventos 'flood' históricos: {len(eventos_flood)}")

# Open the classified raster
with rasterio.open(RUTA_CLASIFICADO) as src:
    # Coordinates of each flood event
    coords = [(geom.x, geom.y) for geom in eventos_flood.geometry]
    # Sample the SAR category at each event location
    eventos_flood["sar_categoria"] = [int(next(src.sample([c]))[0]) for c in coords]

# Labels of the SAR categories
ETIQUETAS_VAL = {0: "Sin señal SAR", 1: "Muy Baja", 2: "Baja", 3: "Moderada", 4: "Alta", 5: "Muy Alta"}
# Add the label to each event
eventos_flood["sar_etiqueta"] = eventos_flood["sar_categoria"].map(ETIQUETAS_VAL)

# Number of events per SAR category
tabla_val = eventos_flood["sar_etiqueta"].value_counts().reindex(
    ["Sin señal SAR", "Muy Baja", "Baja", "Moderada", "Alta", "Muy Alta"]
).fillna(0).astype(int)
# Percentage of events per SAR category
tabla_val_pct = (tabla_val / len(eventos_flood) * 100).round(2)
# Print an empty line
print()
# Print the validation table
print(pd.DataFrame({"n_eventos": tabla_val, "pct": tabla_val_pct}))

In [ ]:
# Days covered by the 9 valid dates (13 days each)
dias_cubiertos = 9 * (2 * 6 + 1)
# Total days in the study period
dias_totales = (pd.Timestamp("2025-12-31") - pd.Timestamp("2015-01-01")).days
# Probability that an event falls by chance on a covered day
prob_azar = dias_cubiertos / dias_totales

# Number of events with some SAR signal
n_con_senal = int((eventos_flood["sar_categoria"] > 0).sum())
# Observed share of events with some SAR signal
tasa_observada = n_con_senal / len(eventos_flood)

# Print the covered days and the chance probability
print(f"Días cubiertos por las 9 fechas válidas (±6 días c/u, sin traslape): ~{dias_cubiertos} de {dias_totales} "
      f"({prob_azar*100:.2f}% del calendario 2015-2025)")
# Print the observed rate
print(f"Tasa observada de eventos con alguna señal SAR: {tasa_observada*100:.2f}%")
# Print the enrichment over chance
print(f"Enriquecimiento vs. azar: {tasa_observada/prob_azar:.1f}x")

## 6. Static map (hexagon aggregation)

In [ ]:
# Province boundaries for the map
province = gpd.read_file(processed_dir / "adm1_province.geojson")

# Colors of the categories
COLORES = {0: "#f0f0f0", 1: "#ffffcc", 2: "#fed976", 3: "#fd8d3c", 4: "#e31a1c", 5: "#800026"}

# Create the figure
fig, ax = plt.subplots(figsize=(8, 10))
# Draw province boundaries
province.boundary.plot(ax=ax, color="black", linewidth=0.3, zorder=1)
# Loop over the categories
for cat in [0, 1, 2, 3, 4, 5]:
    # Hexagons of the category
    sub = hex_gdf[hex_gdf["categoria"] == cat]
    # Skip empty categories
    if len(sub) == 0:
        # Move on to the next unit
        continue
    # Draw the hexagons of the category
    sub.plot(ax=ax, color=COLORES[cat], edgecolor="white", linewidth=0.1, zorder=2)

# Legend entries
handles = [Patch(facecolor=COLORES[c], edgecolor="white", label=ETIQUETAS_HEX[c]) for c in [1, 2, 3, 4, 5]]
# Draw the legend
ax.legend(handles=handles, loc="lower left", fontsize=7,
          title="Susceptibilidad SAR\n(frecuencia media por hexágono)", title_fontsize=7.5, frameon=True)
# Set the map title
ax.set_title(
    "Susceptibilidad a Inundación desde Sentinel-1 (SAR)\n"
    "Agregación hexagonal H3 res.5 · 9 fechas válidas de 15 objetivo (2017/2023/2024)",
    fontsize=11, fontweight="bold",
)
# Hide the axes
ax.set_axis_off()
# Add a source annotation
ax.annotate("EFCI Ecuador · Sentinel-1 SAR 2015-2025",
            xy=(0.01, 0.01), xycoords="axes fraction", fontsize=6, color="gray")
# Adjust the layout
fig.tight_layout()
# Save the map as PNG
fig.savefig(maps_dir / "mapa_sar_flood_susceptibility.png", dpi=150, bbox_inches="tight")
# Log that the map was generated
logger.info("Mapa SAR generado: mapa_sar_flood_susceptibility.png")
# Show the figure
plt.show()

## 7. Export summary